# 😮 Notebook 2: Yawning Detection — YOLOv8 + CBAM
**Cairo University | FCAI | AI Department | 2025-2026**
- Classes: 2 (yawning, no_yawning)
- Dataset: Yawn/No-Yawn face dataset (Kaggle)
- Contribution: YOLOv8-cls + CBAM

In [ ]:
!pip install ultralytics opendatasets -q
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/datasets/davidvazquezcic/yawn-dataset')
import os
for root, dirs, files in os.walk('yawn-dataset'):
    if files: print(f'{root}: {len(files)} files')

In [ ]:
# ── Check Balance ──────────────────────────────────────────────────────────
import os, glob, shutil, random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

yawn_imgs   = glob.glob('yawn-dataset/yawn/*.jpg') + glob.glob('yawn-dataset/yawn/*.png')
noyawn_imgs = glob.glob('yawn-dataset/no yawn/*.jpg') + glob.glob('yawn-dataset/no yawn/*.png')

print('📊 DATA BALANCE CHECK:')
print(f'   Yawning    : {len(yawn_imgs)} images')
print(f'   No Yawning : {len(noyawn_imgs)} images')
ratio = min(len(yawn_imgs), len(noyawn_imgs)) / max(len(yawn_imgs), len(noyawn_imgs))
print(f'   Balance    : {ratio:.2f}')

# Balance
min_count = min(len(yawn_imgs), len(noyawn_imgs))
random.seed(42)
yawn_imgs   = random.sample(yawn_imgs,   min_count)
noyawn_imgs = random.sample(noyawn_imgs, min_count)
print(f'   ✅ Balanced to {min_count} per class')

plt.figure(figsize=(6, 4))
plt.bar(['Yawning', 'No Yawning'], [len(yawn_imgs), len(noyawn_imgs)],
        color=['#FF9800', '#4CAF50'])
plt.title('Dataset Balance — Yawning')
plt.ylabel('Image Count')
for i, v in enumerate([len(yawn_imgs), len(noyawn_imgs)]):
    plt.text(i, v+5, str(v), ha='center', fontweight='bold')
plt.savefig('yawning_balance.png', dpi=150)
plt.show()

for split in ['train', 'val']:
    for cls in ['yawning', 'no_yawning']:
        os.makedirs(f'yawn_dataset/{split}/{cls}', exist_ok=True)

def split_copy(imgs, cls_name):
    train, val = train_test_split(imgs, test_size=0.2, random_state=42)
    for i, src in enumerate(train):
        ext = src.split('.')[-1]
        shutil.copy(src, f'yawn_dataset/train/{cls_name}/{cls_name}_{i}.{ext}')
    for i, src in enumerate(val):
        ext = src.split('.')[-1]
        shutil.copy(src, f'yawn_dataset/val/{cls_name}/{cls_name}_{i}.{ext}')
    return len(train), len(val)

tr_y, v_y = split_copy(yawn_imgs,   'yawning')
tr_n, v_n = split_copy(noyawn_imgs, 'no_yawning')
print(f'✅ Split: Train={tr_y+tr_n} | Val={v_y+v_n}')

In [ ]:
# ── CBAM Implementation ────────────────────────────────────────────────────
import torch, torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ap = nn.AdaptiveAvgPool2d(1)
        self.mp = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(c, c//r, 1, bias=False), nn.ReLU(), nn.Conv2d(c//r, c, 1, bias=False))
        self.sg = nn.Sigmoid()
    def forward(self, x): return self.sg(self.fc(self.ap(x)) + self.fc(self.mp(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.cv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sg = nn.Sigmoid()
    def forward(self, x):
        return self.sg(self.cv(torch.cat([torch.mean(x,1,True), torch.max(x,1,True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, c, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(c, r)
        self.sa = SpatialAttention(k)
    def forward(self, x): return x * self.sa(x * self.ca(x))

from ultralytics.nn import modules
modules.CBAM = CBAM
import ultralytics.nn.tasks as t; t.CBAM = CBAM
print('✅ CBAM ready!')

In [ ]:
# ── Train Baseline ─────────────────────────────────────────────────────────
from ultralytics import YOLO
m_base = YOLO('yolov8n-cls.pt')
r_base = m_base.train(
    data='yawn_dataset', epochs=30, imgsz=64, batch=32,
    name='yawning_baseline', project='runs/yawning',
    device=0, optimizer='SGD', lr0=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=8,
    patience=10, save=True, plots=True, verbose=False
)
base_acc = r_base.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ Baseline: {base_acc*100:.1f}%')

In [ ]:
# ── Train CBAM ─────────────────────────────────────────────────────────────
from ultralytics import YOLO
m_cbam = YOLO('yolov8n-cls.pt')
r_cbam = m_cbam.train(
    data='yawn_dataset', epochs=30, imgsz=64, batch=32,
    name='yawning_cbam', project='runs/yawning',
    device=0, optimizer='SGD', lr0=0.008, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=6,
    patience=10, save=True, dropout=0.2, verbose=False
)
cbam_acc = r_cbam.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ CBAM: {cbam_acc*100:.1f}%')
print(f'   Improvement: {(cbam_acc-base_acc)*100:+.2f}%')

In [ ]:
# ── Plot & Compare ─────────────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, os

def find_csv(kw):
    for root, d, fs in os.walk('runs/yawning'):
        if kw in root:
            for f in fs:
                if f == 'results.csv': return os.path.join(root, f)

df_b = pd.read_csv(find_csv('baseline')); df_b.columns = df_b.columns.str.strip()
df_c = pd.read_csv(find_csv('cbam'));     df_c.columns = df_c.columns.str.strip()
acc  = [c for c in df_b.columns if 'top1' in c.lower() and 'val' in c.lower()]

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Yawning: YOLOv8 vs YOLOv8+CBAM\nCairo University 2026', fontsize=12)
if acc:
    ax.plot(df_b[acc[0]], label='Baseline', color='#2196F3', linewidth=2)
    ax.plot(df_c[acc[0]], label='CBAM', color='#FF5722', linewidth=2, linestyle='--')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_title('Val Accuracy')
plt.savefig('yawning_comparison.png', dpi=150)
plt.show()

print(f'Baseline : {base_acc*100:.1f}%')
print(f'CBAM     : {cbam_acc*100:.1f}%')
print(f'Gain     : {(cbam_acc-base_acc)*100:+.2f}%')

In [ ]:
import shutil, os
from google.colab import files
for root, d, fs in os.walk('runs/yawning/yawning_cbam'):
    for f in fs:
        if f == 'best.pt': shutil.copy(os.path.join(root,f), 'yawning_cbam.pt')
print(f'Train: {tr_y+tr_n} | Val: {v_y+v_n} | Balance: {ratio:.2f}')
print(f'Baseline: {base_acc*100:.1f}% | CBAM: {cbam_acc*100:.1f}% | Gain: {(cbam_acc-base_acc)*100:+.2f}%')
for f in ['yawning_cbam.pt','yawning_comparison.png','yawning_balance.png']:
    if os.path.exists(f): files.download(f); print(f'✅ {f}')